# 过拟合修正：有时间边界的模型检查点选择

保留已有神经网络与训练轨迹，仅用当时可知的前两年验证结果选择训练预算。零重训、不增加手工交易规则。计划在任何新回放前固定，旧研究保持不变。

## 问题与固定实验

旧 Direct/PIT 的 outer 训练存在标签跨截止日成熟问题；固定轮数或最小训练损失也不等于验证选择。较新的 constrained 模型已具备严格成熟标签和完整费用，适合作为本轮控制基线。只复用 price / price_vol 的 42 条已保存训练轨迹（每条 0–120），不重新训练。

每年 Y 从前两年、且在 Y 年初已经成熟的结果选择预算 15/30/60/120 或显式 SOXX 空模型。所有三个 seed 均保留，集合自己的实际持仓独立结算。2016/17 为共同固定 120 轮预热，主要评价 2018–2022。空模型与轮数选择构成一项选择机制，不能把任何改善单独归为早停。历史均已复用，禁止宣称全新留出集。

In [1]:
import os, sys, json, hashlib
from pathlib import Path
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "SPEC.md").exists())
EVIDENCE_ROOT = Path(os.environ.get("SOXL_CANONICAL_EVIDENCE_ROOT", str(ROOT))).resolve()
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from qlib.data import D
from src.data.constrained_net_excess import load_constrained_inputs
from src.workflow.anchor_checkpoint_feasibility import checked_parent_load
from src.workflow.constrained_net_excess import training_spec
from src.workflow.benchmark_anchor_utility import replay_models, independent_path_audit
from src.workflow.stable_excess_integration import benchmark_path
from src.analysis.integrated_allocation import paired_metrics, annual_paired_metrics, stationary_bootstrap_excess, native_allocation_reconciliation
from src.analysis.continuous_target_adapter import native_open_reconciliation
from src.data.integrated_allocation import frame_sha256
from src.model.purged_checkpoint_selection import reconstruct_budget_bundles, select_budget, annual_bundles, VARIANTS, YEARS, OUTER_YEARS, SEEDS, COSTS, BUDGETS
CARD = ROOT / "research/purged_checkpoint_validation_20261008"
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
plan=json.loads((CARD/"plan.json").read_text());binding=json.loads((CARD/"run_binding.json").read_text())
assert binding["plan_sha256"] == sha(CARD/"plan.json")
assert all(sha(ROOT/path)==digest for path,digest in binding["source_sha256"].items())
nb=json.loads((CARD/(CARD.name+".ipynb")).read_text())
code_text="\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"]=="code")
assert hashlib.sha256(code_text.encode()).hexdigest()==binding["notebook_code_sha256"]
assert list(BUDGETS)==plan["selection_candidates"] and list(VARIANTS)==plan["variants"]
PARENT=EVIDENCE_ROOT/"research"/plan["source"]["experiment"]
assert sha(PARENT/"artifact_manifest.json")==plan["source"]["manifest_sha256"]
assert sha(PARENT/"summary.json")==plan["source"]["summary_sha256"]
assert sha(PARENT/"selection_plan.json")==plan["source"]["plan_sha256"]
parent_summary=json.loads((PARENT/"summary.json").read_text());manifest=json.loads((PARENT/"artifact_manifest.json").read_text())
parent_plan=json.loads((PARENT/"selection_plan.json").read_text());spec=training_spec(parent_plan)
# Source loading reconstructs native D.features + DatasetH and verifies the frozen parent identities.
native_bundle=load_constrained_inputs(EVIDENCE_ROOT,parent_plan)
qlib.init(provider_uri=str(EVIDENCE_ROOT/plan["source"]["provider_uri"]),region="us",kernels=1)
calendar=pd.DatetimeIndex(D.calendar(freq="day"))
assert str(calendar[-1].date())==plan["source"]["as_of"]
identities=[]
with R.uri_context(str(PARENT/"qlib_recorder")):
    parent=R.get_recorder(experiment_name=plan["source"]["experiment"],recorder_id=plan["source"]["recorder_id"])
    assert parent.client.get_run(parent.id).info.status=="FINISHED"
    def checked(name):
        value,identity=checked_parent_load(parent,PARENT,manifest,name)
        identities.append(identity)
        return value
    hashes=checked("inputs/source_hashes.pkl")
    assert all(sha(EVIDENCE_ROOT/path)==digest for path,digest in hashes.items())
    folds=checked("inputs/development_fold_inputs.pkl")
    original=checked("models/development.pkl")
    forecasts=checked("inputs/forecasts.pkl")
    frozen_paths=checked("analysis/portfolio_replays.pkl")
    audits={(variant,year,seed):checked(f"fit_checkpoints/{variant}/{year}/{seed}/training_audit.pkl")
            for variant in VARIANTS for year in YEARS for seed in SEEDS}
pd.testing.assert_frame_equal(forecasts,native_bundle["forecasts"],check_exact=True)
for key,fold in folds.items():
    old=native_bundle["parent_dev_folds"][key]
    assert fold["scaler"]==old["scaler"]
    np.testing.assert_array_equal(fold["features"],old["features"])
    np.testing.assert_array_equal(fold["returns"],old["returns"])
frame=forecasts.loc[forecasts.index.year.isin(YEARS)].copy()
assert np.isfinite(frame[["soxl_return","soxx_return"]].to_numpy()).all()
print({"source_recorder":parent.id,"audited_seed_fits":len(audits),"saved_epoch_states":sum(len(x["epochs"]) for x in audits.values()),"new_fits":0,"evaluation_source_rows":len(frame)})

2026/10/08 11:52:17 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[8037:MainThread](2026-10-08 11:52:19,532) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[8037:MainThread](2026-10-08 11:52:19,533) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[8037:MainThread](2026-10-08 11:52:19,533) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/published_futu_affine_20260925_v1')}


...

[8037:MainThread](2026-10-08 11:52:19,715) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


[8037:MainThread](2026-10-08 11:52:21,391) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[8037:MainThread](2026-10-08 11:52:21,392) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[8037:MainThread](2026-10-08 11:52:21,392) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/published_futu_affine_20260925_v1')}


{'source_recorder': '300530137db54b6c807c39ebb9e8c331', 'audited_seed_fits': 42, 'saved_epoch_states': 5082, 'new_fits': 0, 'evaluation_source_rows': 1762}


## 复原各预算的原训练选择

预算上限不等于最终选中 epoch：保留预算内第一个严格改善训练 F、且满足原始训练风险门的状态；与父实验完全一致。每个集合重新按其自身持仓计算两档成本和风险。120 预算的模型张量与原研究必须逐项完全相同，否则停止。

In [2]:
bundles, checkpoint_index = reconstruct_budget_bundles(folds,audits,spec=spec)
for variant in VARIANTS:
    for year in YEARS:
        restored=bundles[(variant,120)][year];saved=original[variant][year]
        assert restored["scaler"]==saved["scaler"]
        assert [m["tensor_sha256"] for m in restored["ensemble_models"]]==[m["tensor_sha256"] for m in saved["ensemble_models"]]
        assert {s:m["tensor_sha256"] for s,m in restored["seed_models"].items()}=={s:m["tensor_sha256"] for s,m in saved["seed_models"].items()}
print({"year_budget_bundles":sum(len(v) for v in bundles.values()),"seed_budget_records":len(checkpoint_index),"120_budget_identity":"exact"})

{'year_budget_bundles': 70, 'seed_budget_records': 210, '120_budget_identity': 'exact'}


## 仅用过去的验证结果选择下一年度预算

每个验证候选使用自己的两年持仓路径，从验证起点现金开始，跨年不重置。严格删去在选择时尚未成熟的尾部，不压缩中间缺失日期。下一年度或更远收益不参与评分；测试包括对这些未来值的任意扰动。SOXX 是第五个明确登记的模型候选，评分与自身比较为 0。

In [3]:
selected,score_tables,selection_audits={},[],[]
for variant in VARIANTS:
    selected[variant]={}
    for year in OUTER_YEARS:
        budget,scores,audit=select_budget(frame,bundles,variant=variant,outer_year=year)
        learned=scores.loc[scores.budget.gt(0)].sort_values(["selection_score","budget"],ascending=[False,True]).iloc[0]
        audit.update(best_learned_budget=int(learned.budget),best_learned_score=float(learned.selection_score))
        selected[variant][year]=budget;score_tables.append(scores);selection_audits.append(audit)
selection=pd.DataFrame(selection_audits);candidate_scores=pd.concat(score_tables,ignore_index=True)
selection[["variant","outer_year","selected_budget","best_learned_budget","validation_rows","purged_tail_rows","label_available_at_max","selection_cutoff"]]

,variant,outer_year,selected_budget,best_learned_budget,validation_rows,purged_tail_rows,label_available_at_max,selection_cutoff
0,price,2018,120,120,501,2,2017-12-29T14:30:00+00:00,2018-01-01T00:00:00-05:00
1,price,2019,0,15,500,2,2018-12-31T14:30:00+00:00,2019-01-01T00:00:00-05:00
2,price,2020,0,15,501,2,2019-12-31T14:30:00+00:00,2020-01-01T00:00:00-05:00
3,price,2021,0,15,503,2,2020-12-31T14:30:00+00:00,2021-01-01T00:00:00-05:00
4,price,2022,0,15,503,2,2021-12-31T14:30:00+00:00,2022-01-01T00:00:00-05:00
5,price_vol,2018,120,120,501,2,2017-12-29T14:30:00+00:00,2018-01-01T00:00:00-05:00
6,price_vol,2019,0,15,500,2,2018-12-31T14:30:00+00:00,2019-01-01T00:00:00-05:00
7,price_vol,2020,0,15,501,2,2019-12-31T14:30:00+00:00,2020-01-01T00:00:00-05:00
8,price_vol,2021,0,15,503,2,2020-12-31T14:30:00+00:00,2021-01-01T00:00:00-05:00
9,price_vol,2022,0,15,503,2,2021-12-31T14:30:00+00:00,2022-01-01T00:00:00-05:00


## 完整连续回放与基线复现

baseline 和 validation_selected 都从 2016 年同一现金起点开始，2016/17 使用相同原 120 预算模型。2018 起才分化；不平均 seed 的收益，也不在年度重置持仓。原完整 baseline 净收益、实际换手和持仓必须复原。

In [4]:
paths,benchmarks,independent={}, {}, {}
selected_annual={v:annual_bundles(bundles,variant=v,selected_by_year=selected[v]) for v in VARIANTS}
for cost in COSTS:benchmarks[cost]=benchmark_path(frame,cost_bps=cost)
for variant in VARIANTS:
    for method,annual in (("fixed120",bundles[(variant,120)]),("validation_selected",selected_annual[variant])):
        for cost in COSTS:
            key=(variant,method,cost)
            path=replay_models(frame,annual,variant=variant,cost_bps=cost)
            independent[key]=independent_path_audit(frame,path,cost_bps=cost)
            paths[key]=path
            if method=="fixed120":
                saved=frozen_paths.loc[(frozen_paths.variant==variant)&(frozen_paths.seed=="ensemble")&(frozen_paths.segment=="development")&(frozen_paths.cost_bps==cost)].reset_index(drop=True)
                assert len(saved)==len(path)
                for column in ("strategy_return","turnover","fees","soxl_weight","soxx_weight","cash_weight"):
                    np.testing.assert_allclose(path[column],saved[column],rtol=0,atol=1e-12)
                assert path[["signal_date","trade_date","exit_date"]].equals(saved[["signal_date","trade_date","exit_date"]])
            else:
                warm=path.signal_date.dt.year.lt(2018)
                pd.testing.assert_frame_equal(path.loc[warm],paths[(variant,"fixed120",cost)].loc[warm],check_exact=True)
failure=json.loads((CARD/"pre_native_failure.json").read_text())
for item in failure["frozen_scored_paths"]:
    key=(item["variant"],item["method"],item["cost_bps"])
    assert frame_sha256(paths[key])==item["path_sha256"], "blocked: scored path changed during native-only repair"
assert {v:{str(y):b for y,b in value.items()} for v,value in selected.items()}==failure["selected_budgets"]
print({"replay_paths":len(paths),"source120_reproduction":"passed","shared_warmup":"exact","independent_reconciliation":"passed"})

{'replay_paths': 8, 'source120_reproduction': 'passed', 'shared_warmup': 'exact', 'independent_reconciliation': 'passed'}


## 原生 Qlib 对账与 Recorder

所有 8 条完整路径由原生 Qlib Backtest/Exchange/Position 重新结算；按下一开盘标记对齐策略的 open-to-open 回报，不把收盘报告混为同口径。原数据和历史卡不改，新证据存入独立 Recorder。

In [5]:
native={}
for key,path in paths.items():
    native[key]=native_allocation_reconciliation(frame.reset_index(drop=True),path,
        provider_uri=EVIDENCE_ROOT/plan["source"]["provider_uri"],cost_bps=key[2],native_reconciler=native_open_reconciliation)
    assert native[key]["audit"]["status"]=="passed"
print({str(k):v["audit"] for k,v in native.items()})

[8037:MainThread](2026-10-08 11:52:55,184) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[8037:MainThread](2026-10-08 11:52:55,185) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[8037:MainThread](2026-10-08 11:52:55,185) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/published_futu_affine_20260925_v1')}


[8037:MainThread](2026-10-08 11:52:55,196) INFO - qlib.backtest caller - [__init__.py:93] - Create new exchange


[8037:MainThread](2026-10-08 11:52:55,200) WARNING - qlib.online operator - [exchange.py:226] - factor.day.bin file not exists or factor contains `nan`. Order using adjusted_price.


...

[8037:MainThread](2026-10-08 11:53:03,347) WARNING - qlib.online operator - [exchange.py:226] - factor.day.bin file not exists or factor contains `nan`. Order using adjusted_price.


[8037:MainThread](2026-10-08 11:53:03,354) WARNING - qlib.data - [data.py:665] - load calendar error: freq=day, future=True; return current calendar!


[8037:MainThread](2026-10-08 11:53:03,354) WARNING - qlib.data - [data.py:668] - You can get future calendar by referring to the following document: https://github.com/microsoft/qlib/blob/main/scripts/data_collector/contrib/README.md


[8037:MainThread](2026-10-08 11:53:03,357) WARNING - qlib.BaseExecutor - [executor.py:121] - `common_infra` is not set for <qlib.backtest.executor.SimulatorExecutor object at 0x13b17a780>


{"('price', 'fixed120', 10.0)": {'status': 'passed', 'sessions': 1762, 'start': '2016-01-05', 'end': '2023-01-03', 'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/published_futu_affine_20260925_v1', 'cost_bps': 10.0, 'initial_cash': 100000.0, 'maximum_session_return_error': 5.08653617004029e-08, 'tolerance': 1e-07, 'native_api': 'qlib.backtest.backtest + SimulatorExecutor + Exchange + Account + Position', 'position_type': 'src.backtest.qlib_continuous_position.QlibContinuousPosition', 'position_precision_fix': 'Every positive fractional-sale residual remains owned; no relative isclose liquidation.', 'report_mark': 'native same-day close; next-open reconciliation marks native positions through Qlib D.features', 'sealed_stage1_code_changed': False, 'qlib_api_reference': 'https://qlib.readthedocs.io/en/latest/reference/api.html#qlib.backtest.backtest', 'target_roundoff_adapter': {'scope': 'native adapter only; original accounting replay and model selection are unchanged', 'affecte

## 固定评价与解释边界

主评价仅为 2018–2022；全 2016–2022 是含共同预热的补充。两模型族、两档成本、全部年份均保留。不按结果挑族、年份或 seed，也不调整风险与收益门。区块区间仅描述复用历史，未覆盖模型搜索和未来分布变化的不确定性。

In [6]:
comparison_rows,annual_rows,bootstrap_rows=[],[],[]
for scope,years in (("primary_2018_2022",OUTER_YEARS),("full_2016_2022_with_shared_warmup",YEARS)):
    for variant in VARIANTS:
        for cost in COSTS:
            mask=benchmarks[cost].signal_date.dt.year.isin(years)
            benchmark=benchmarks[cost].loc[mask]
            for method in ("fixed120","validation_selected"):
                path=paths[(variant,method,cost)].loc[mask]
                metrics=paired_metrics(path,benchmark)
                comparison_rows.append({"scope":scope,"variant":variant,"method":method,"cost_bps":cost,**metrics})
                annual_rows.append(annual_paired_metrics(path,benchmark).assign(scope=scope,variant=variant,method=method,cost_bps=cost))
            if scope=="primary_2018_2022":
                candidate=paths[(variant,"validation_selected",cost)].loc[mask]
                reference=paths[(variant,"fixed120",cost)].loc[mask]
                for block in plan["uncertainty"]["blocks"]:
                    stats=stationary_bootstrap_excess(candidate,reference,mean_block=block,
                        resamples=plan["uncertainty"]["resamples"],alpha=.05/plan["uncertainty"]["two_variant_family_adjustment"],seed=plan["uncertainty"]["seed"])
                    bootstrap_rows.append({"variant":variant,"cost_bps":cost,**stats})
comparison=pd.DataFrame(comparison_rows);annual=pd.concat(annual_rows,ignore_index=True);bootstrap=pd.DataFrame(bootstrap_rows)
comparison.loc[(comparison.scope=="primary_2018_2022")&(comparison.cost_bps==25),["variant","method","return_pct","relative_wealth","max_drawdown_pct","cvar5_pct","turnover"]]

,variant,method,return_pct,relative_wealth,max_drawdown_pct,cvar5_pct,turnover
2,price,fixed120,103.203122,0.958475,-46.102080,-4.596456,41.859696
3,price,validation_selected,109.978633,0.990434,-47.373291,-4.727658,4.981578
6,price_vol,fixed120,107.549331,0.978975,-46.527776,-4.667181,45.098825
7,price_vol,validation_selected,110.834948,0.994473,-47.373291,-4.749180,5.541497


In [7]:
bootstrap[["variant","cost_bps","mean_block","annualized_log_excess","annualized_lower_bound","annualized_upper_bound","lower_bound_positive"]]

,variant,cost_bps,mean_block,annualized_log_excess,annualized_lower_bound,annualized_upper_bound,lower_bound_positive
0,price,10.0,20,-0.004508,-0.037675,0.028815,False
1,price,10.0,60,-0.004508,-0.036054,0.026481,False
2,price,25.0,20,0.006565,-0.026628,0.039690,False
3,price,25.0,60,0.006565,-0.024559,0.037569,False
4,price_vol,10.0,20,-0.008734,-0.043290,0.026770,False
5,price_vol,10.0,60,-0.008734,-0.040508,0.022516,False
6,price_vol,25.0,20,0.003144,-0.031060,0.038496,False
7,price_vol,25.0,60,0.003144,-0.028042,0.036192,False


In [8]:
tables={"checkpoint_index.csv":checkpoint_index,"selection_audit.csv":selection,"candidate_scores.csv":candidate_scores,
        "backtest_benchmark_comparison.csv":comparison,"annual_metrics.csv":annual,"bootstrap_comparison.csv":bootstrap}
for filename,table in tables.items():table.to_csv(CARD/filename,index=False)
summary={"experiment":CARD.name,"record_type":"purged_prior_validation_of_frozen_optimizer_budgets","provider_uri":plan["source"]["provider_uri"],"region":"us","market":"all","as_of":plan["source"]["as_of"],
    "source_recorder_id":plan["source"]["recorder_id"],"source_manifest_sha256":plan["source"]["manifest_sha256"],"source_artifact_identities":identities,
    "source_sha256":binding["source_sha256"],"plan_sha256":sha(CARD/"plan.json"),"binding_sha256":sha(CARD/"run_binding.json"),
    "market_model_fits":0,"saved_seed_fits":len(audits),"saved_epoch_states":sum(len(a["epochs"]) for a in audits.values()),"all_three_seeds_used":True,
    "baseline120_checkpoint_identity":"exact","baseline120_replay_max_atol":1e-12,"shared_warmup_2016_2017":"exact","native_reconciled_paths":len(native),
    "primary_years":list(OUTER_YEARS),"validation_choices":{v:{str(y):int(b) for y,b in x.items()} for v,x in selected.items()},
    "null_selection_count":int(selection.selected_budget.eq(0).sum()),"total_selections":len(selection),
    "fresh_holdout":False,"post2022_performance_evaluated":False,"selection_design_proves_overfit_solved":False,
    "promotion_decision":"not_promoted_reused_history_diagnostic","production_change":False,"objective_achieved":False,
    "artifact_policy":{"policy":"slim_card_recorder_first","workflow_config":"workflow.yaml","recorder_only_artifacts":["models/budget_bundles.pkl","models/selected_annual.pkl","analysis/portfolio_replays.pkl","portfolio_analysis/native_reconciliation.pkl"]},
    "sidecars":list(tables)+["plan.json","run_binding.json","source_review.json","pre_native_failure.json","conclusion.md"],"qlib_docs_reference":["https://qlib.readthedocs.io/en/latest/component/data.html","https://qlib.readthedocs.io/en/latest/component/recorder.html"],"limitations":plan["limitations"]}
uri=str((CARD/"qlib_recorder").resolve())
with R.start(experiment_name=CARD.name,uri=uri):
    rec=R.get_recorder()
    summary.update(recorder_id=rec.id,experiment_id=rec.experiment_id,recorder_uri=str((CARD/"qlib_recorder").relative_to(ROOT)),recorder_relative_dir=str(Path(rec.get_local_dir()).relative_to(CARD/"qlib_recorder")))
    R.log_params(plan_sha256=sha(CARD/"plan.json"),source_recorder_id=plan["source"]["recorder_id"],new_fits=0)
    R.log_metrics(native_reconciled_paths=len(native),null_selections=summary["null_selection_count"])
    R.save_objects(artifact_path="models",**{"budget_bundles.pkl":bundles,"selected_annual.pkl":selected_annual})
    R.save_objects(artifact_path="analysis",**{"portfolio_replays.pkl":paths,"benchmarks.pkl":benchmarks,"tables.pkl":tables,"independent_reconciliation.pkl":independent,"summary.pkl":summary})
    R.save_objects(artifact_path="portfolio_analysis",**{"native_reconciliation.pkl":native})
    R.save_objects(artifact_path="inputs",**{"source_identity.pkl":identities,"plan.pkl":plan,"source_audit.pkl":native_bundle["source_audit"]})
    for path in binding["source_sha256"]:rec.log_artifact(str(ROOT/path),artifact_path="source/"+str(Path(path).parent))
    for filename in [*tables,"plan.json","workflow.yaml","run_binding.json","source_review.json","pre_native_failure.json"]:rec.log_artifact(str(CARD/filename),artifact_path="card")
(CARD/"summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2,allow_nan=False)+"\n")
lines=["# 过拟合修正实验结论", "完成两个既有神经网络模型族的过去两年验证选训练预算；零重训，不加手工交易规则。保留全部三个种子及原训练风险约束，显式 SOXX 空模型作为第五个候选。", "主要诊断为2018–2022，2016/17共同固定120预算预热，不作为五次外层选择的新验证；所有历史均已复用。"]
for variant in VARIANTS:
    lines.append(f"\n{variant}，每侧25bp：")
    for method in ("fixed120","validation_selected"):
        row=comparison.loc[(comparison.scope=="primary_2018_2022")&(comparison.variant==variant)&(comparison.method==method)&(comparison.cost_bps==25)].iloc[0]
        lines.append(f"- {method}：收益{row.return_pct:.2f}%，终值/SOXX={row.relative_wealth:.4f}，MaxDD={row.max_drawdown_pct:.2f}%，CVaR5={row.cvar5_pct:.2f}%，换手{row.turnover:.2f}。")
lines += [f"\n10个年度模型选择中，{summary['null_selection_count']}次选SOXX空模型；具体预算和成熟截止见selection_audit.csv。预算上限不等于被保留的具体epoch。若收益改善来自选择空模型，只说明当时验证没有支持主动模型，不能说模型学出了更好的alpha。",
"四十二条原训练轨迹的5082个状态按hash核验；120预算原模型及完整收益/持仓复原，共同预热完全相同。所有8条完整路径通过原生Qlib对账，独立资金结算亦通过。",
"两种区块长度的adaptive-minus-fixed区间见bootstrap_comparison.csv，逐年结果见annual_metrics.csv。区间不涵盖历史反复研究与模型选择不确定性；不把最好的模型族单独挑出来宣称成功。",
"不晋升、不改生产。此次修正的是选择过程的时间边界，尚未证明过拟合已解决。旧Direct/PIT的跨截止标签、推理标签依赖、训练损失选checkpoint和损失/state错位仍作为旧版限制保留；新实验使用已严格校验的canonical Dataset路径，没有重签旧结果。",
"下一步必须冻结完整训练与选模协议，用连续新数据确认；若继续历史研究，应一次只测试一个已登记的正则化机制，并报告全部失败，不增加网络或规则网格。",
f"Qlib Recorder：{rec.id}；源Recorder：{parent.id}；as_of={plan['source']['as_of']}。"]
conclusion="\n\n".join(lines)+"\n";(CARD/"conclusion.md").write_text(conclusion)
with R.uri_context(uri):
    recorder=R.get_recorder(experiment_name=CARD.name,recorder_id=rec.id)
    for filename in ("summary.json","conclusion.md"):recorder.log_artifact(str(CARD/filename),artifact_path="card")
print(conclusion)

[8037:MainThread](2026-10-08 11:53:05,280) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name purged_checkpoint_validation_20261008.


[8037:MainThread](2026-10-08 11:53:05,282) INFO - qlib.workflow - [exp.py:258] - Experiment 336206974761097994 starts running ...


[8037:MainThread](2026-10-08 11:53:05,359) INFO - qlib.workflow - [recorder.py:345] - Recorder 4b6cac2e05a648788e3c6732563a1f25 starts running under Experiment 336206974761097994 ...


[8037:MainThread](2026-10-08 11:53:06,677) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


# 过拟合修正实验结论

完成两个既有神经网络模型族的过去两年验证选训练预算；零重训，不加手工交易规则。保留全部三个种子及原训练风险约束，显式 SOXX 空模型作为第五个候选。

主要诊断为2018–2022，2016/17共同固定120预算预热，不作为五次外层选择的新验证；所有历史均已复用。


price，每侧25bp：

- fixed120：收益103.20%，终值/SOXX=0.9585，MaxDD=-46.10%，CVaR5=-4.60%，换手41.86。

- validation_selected：收益109.98%，终值/SOXX=0.9904，MaxDD=-47.37%，CVaR5=-4.73%，换手4.98。


price_vol，每侧25bp：

- fixed120：收益107.55%，终值/SOXX=0.9790，MaxDD=-46.53%，CVaR5=-4.67%，换手45.10。

- validation_selected：收益110.83%，终值/SOXX=0.9945，MaxDD=-47.37%，CVaR5=-4.75%，换手5.54。


10个年度模型选择中，8次选SOXX空模型；具体预算和成熟截止见selection_audit.csv。预算上限不等于被保留的具体epoch。若收益改善来自选择空模型，只说明当时验证没有支持主动模型，不能说模型学出了更好的alpha。

四十二条原训练轨迹的5082个状态按hash核验；120预算原模型及完整收益/持仓复原，共同预热完全相同。所有8条完整路径通过原生Qlib对账，独立资金结算亦通过。

两种区块长度的adaptive-minus-fixed区间见bootstrap_comparison.csv，逐年结果见annual_metrics.csv。区间不涵盖历史反复研究与模型选择不确定性；不把最好的模型族单独挑出来宣称成功。

不晋升、不改生产。此次修正的是选择过程的时间边界，尚未证明过拟合已解决。旧Direct/PIT的跨截止标签、推理标签依赖、训练损失选checkpoint和损失/state错位仍作为旧版限制保留；新实验使用已严格校验的canonical Dataset路径，没有重签旧结果。

下一步必须冻结完整训练与选模协议，用连续新数据确认；若继续历史研究，应一次只测试一

# 过拟合修正实验结论

完成两个既有神经网络模型族的过去两年验证选训练预算；零重训，不加手工交易规则。保留全部三个种子及原训练风险约束，显式 SOXX 空模型作为第五个候选。

主要诊断为2018–2022，2016/17共同固定120预算预热，不作为五次外层选择的新验证；所有历史均已复用。


price，每侧25bp：

- fixed120：收益103.20%，终值/SOXX=0.9585，MaxDD=-46.10%，CVaR5=-4.60%，换手41.86。

- validation_selected：收益109.98%，终值/SOXX=0.9904，MaxDD=-47.37%，CVaR5=-4.73%，换手4.98。


price_vol，每侧25bp：

- fixed120：收益107.55%，终值/SOXX=0.9790，MaxDD=-46.53%，CVaR5=-4.67%，换手45.10。

- validation_selected：收益110.83%，终值/SOXX=0.9945，MaxDD=-47.37%，CVaR5=-4.75%，换手5.54。


10个年度模型选择中，8次选SOXX空模型；具体预算和成熟截止见selection_audit.csv。预算上限不等于被保留的具体epoch。若收益改善来自选择空模型，只说明当时验证没有支持主动模型，不能说模型学出了更好的alpha。

四十二条原训练轨迹的5082个状态按hash核验；120预算原模型及完整收益/持仓复原，共同预热完全相同。所有8条完整路径通过原生Qlib对账，独立资金结算亦通过。

两种区块长度的adaptive-minus-fixed区间见bootstrap_comparison.csv，逐年结果见annual_metrics.csv。区间不涵盖历史反复研究与模型选择不确定性；不把最好的模型族单独挑出来宣称成功。

不晋升、不改生产。此次修正的是选择过程的时间边界，尚未证明过拟合已解决。旧Direct/PIT的跨截止标签、推理标签依赖、训练损失选checkpoint和损失/state错位仍作为旧版限制保留；新实验使用已严格校验的canonical Dataset路径，没有重签旧结果。

下一步必须冻结完整训练与选模协议，用连续新数据确认；若继续历史研究，应一次只测试一个已登记的正则化机制，并报告全部失败，不增加网络或规则网格。

Qlib Recorder：4b6cac2e05a648788e3c6732563a1f25；源Recorder：300530137db54b6c807c39ebb9e8c331；as_of=2026-09-25。

## 两档成本与过拟合判断

不能只报告25bp改善：10bp下，price收益从116.37%降至111.55%，price_vol从122.08%降至112.59%。25bp下两个选择版本虽优于固定120，但终值仍分别只有SOXX的0.9904/0.9945；全部8个 adaptive-minus-fixed 区块区间跨零。选择版本最大回撤为−47.37%，比对应固定120版本更差，不能描述为风险改善。

15/30/60预算没有被选中；两个族在2018年都选120，2019–2022都选显式SOXX。主要变化来自拒绝没有历史验证优势的主动模型，而不是模型通过较短训练学会了新的alpha。高成本下换手从41.86/45.10降至4.98/5.54解释了部分收益差异；选择结果不构成未来超额保证。

现阶段的解决顺序应是：先保证严格的标签成熟和预测可用性；训练/验证/外层的职责分离；固定有限候选与全部种子，容许基准模型胜出；再用冻结协议获取真正新的连续验证。L2、降维或降低网络容量只能作为分别登记的对照，不能弥补反复查看同一段历史造成的选择偏差。旧实现问题与本轮已实施范围见overfit_audit.csv。

方法参考：[scikit-learn嵌套选择说明](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html)。本实验使用先后时间而非随机K折；该资料说明选择与评估分离的原则，不为本次金融结果背书。

## 验证与归档

95项相关测试通过，包括未来标签/特征扰动不改变选择、严格成熟时刻、交易日缺口、checkpoint身份、空模型对照和原生Qlib合成回放。要求的旧Qlib套件8通过、61个既有缺失卡片失败；默认全量pytest仍有15个既有归档测试同名导入冲突，不能声称全仓通过。

第一次执行在创建新Recorder前被原生接口的负权重检查阻断。原结算在精确SOXX切换时产生至多约5.71e-17负现金；仅native副本的4行作浮点边界规范化，最大改变量2.22e-16，原1e-7对账容差未变。模型、选择和全部8条原评分路径hash保持不变；失败绑定/原notebook和失败日志在Recorder中保留。

为使主notebook易审阅，仅删除重复stderr行及临时进度组件，未改代码、研究输出或执行计数；完整原始已执行notebook仍存Recorder。最终中文结论为执行后依据保存小表撰写的解释。仅新Recorder元数据迁移到主仓，详见relocation.json；逐文件来源、用途、范围和hash见verification_manifest.json。

## 短训练预算的可行性限制

本次不能把较短训练直接当作有效正则化对照：15和30更新预算下，42/42个种子训练保留状态都是SOXX空模型；60预算下也有36/42个空模型，120预算才全部找到训练可行的正F状态。因此短预算同时改变了“是否已经找到可行解”，并未充分分离过拟合和优化不足。短预算未入选不能证明早停无效，退回基准也不能说明学到了稳健alpha。后续正则化实验应先确认每个对照能达到可行训练状态，再比较严格内层验证与后续期差距；仍不得用本轮结果追加预算网格。详见budget_feasibility_summary.csv。
